# Agente de Connect6 — MCTS (UCB1) + Aprendizaje por Refuerzo Profundo

**Proyecto 2 — Inteligencia Artificial**

Este cuaderno es **auto-contenido**: entrena la red neuronal profunda del agente
en GPU y demuestra el agente completo jugando. No necesita clonar el repositorio
(aunque también funciona si lo clonas).

## Arquitectura del agente

El algoritmo principal es **Monte Carlo Tree Search (MCTS)** (cap. 6 de Russell &
Norvig), con sus cuatro fases:

1. **Selección** con la política **UCB1 / UCT** (pág. 209 del libro):
   $$\text{UCB1}(hijo) = \underbrace{\frac{W(hijo)}{N(hijo)}}_{\text{explotación}} + \; C\,\underbrace{\sqrt{\frac{\ln N(padre)}{N(hijo)}}}_{\text{exploración}}$$
2. **Expansión** de un nodo hijo no explorado.
3. **Simulación (playout)**: aquí entra el **aprendizaje por refuerzo**. En vez de
   jugar al azar, cada jugador coloca la ficha que le **recomienda una red neuronal
   profunda** entrenada con Q-learning (idea de *AlphaGo*).
4. **Retropropagación** del resultado por el árbol.

Además, el agente usa **estrategias propias de Connect6**: apertura al centro,
detección de **victoria inmediata** y **bloqueo** de la amenaza del rival, y
**poda de vecindad** (sólo considera casillas cercanas a fichas existentes).

> Ejecuta las celdas en orden con **Entorno de ejecución → Ejecutar todo**.


## 0. Verificar la GPU
Menú **Entorno de ejecución → Cambiar tipo de entorno → GPU T4**.

In [ ]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPUs disponibles:", gpus if gpus else "NINGUNA (ve a Entorno de ejecución → Cambiar tipo de entorno → GPU)")

## 1. Lógica del tablero de Connect6
Tablero 19×19, reglas de victoria (6 en línea) y utilidades para el MCTS (clonar, candidatos por vecindad, chequeo rápido de victoria por punto).

In [ ]:
import numpy as np

class Connect6Board:
    def __init__(self):
        # 0: Vacío, 1: Negras (Juega primero, 1 ficha en el primer turno), 2: Blancas
        self.size = 19
        self.grid = np.zeros((self.size, self.size), dtype=np.int8)
        self.turn_count = 0

    def clone(self):
        """Copia profunda y barata del tablero (para la simulación del MCTS)."""
        new_board = Connect6Board()
        new_board.grid = self.grid.copy()
        new_board.turn_count = self.turn_count
        return new_board

    def get_valid_moves(self):
        """Retorna una lista de tuplas (x, y) con las casillas vacías."""
        return list(zip(*np.where(self.grid == 0)))

    def get_candidate_moves(self, radius=1):
        """
        Poda de relevancia (imprescindible en tableros 19x19): sólo considera
        casillas vacías que estén a `radius` de alguna ficha ya colocada.
        Si el tablero está vacío, devuelve únicamente el centro.
        Reduce el factor de ramificación de 361 a unas pocas decenas.
        """
        occupied = np.argwhere(self.grid != 0)
        if len(occupied) == 0:
            c = self.size // 2
            return [(c, c)]

        candidates = set()
        n = self.size
        for (ox, oy) in occupied:
            for dx in range(-radius, radius + 1):
                for dy in range(-radius, radius + 1):
                    x, y = int(ox) + dx, int(oy) + dy
                    if 0 <= x < n and 0 <= y < n and self.grid[x, y] == 0:
                        candidates.add((x, y))
        return list(candidates)

    def is_full(self):
        """Verifica si el tablero está lleno."""
        return len(self.get_valid_moves()) == 0

    def wins_at(self, player_id, x, y):
        """
        Chequeo RÁPIDO: ¿colocar `player_id` en (x, y) forma una línea de >=6?
        Sólo examina las 4 direcciones que pasan por (x, y), en vez de escanear
        todo el tablero. Se usa para detectar victoria/bloqueo inmediato y como
        chequeo de terminalidad durante las simulaciones del MCTS.
        """
        x, y = int(x), int(y)
        if self.grid[x, y] != 0 and self.grid[x, y] != player_id:
            return False
        n = self.size
        directions = [(1, 0), (0, 1), (1, 1), (1, -1)]
        for dx, dy in directions:
            count = 1  # la ficha hipotética en (x, y)
            # hacia adelante
            i, j = x + dx, y + dy
            while 0 <= i < n and 0 <= j < n and self.grid[i, j] == player_id:
                count += 1
                i += dx
                j += dy
            # hacia atrás
            i, j = x - dx, y - dy
            while 0 <= i < n and 0 <= j < n and self.grid[i, j] == player_id:
                count += 1
                i -= dx
                j -= dy
            if count >= 6:
                return True
        return False

    def get_state(self, player_id):
        """
        Retorna el estado del tablero desde la perspectiva del jugador.
        1.0 para sus fichas, -1.0 para las del oponente, 0.0 vacío.
        Con forma (19, 19, 1) para Keras CNN.
        """
        state = np.zeros((self.size, self.size, 1), dtype=np.float32)
        state[self.grid == player_id, 0] = 1.0
        state[(self.grid != 0) & (self.grid != player_id), 0] = -1.0
        return state

    def apply_move(self, player_id, moves):
        """
        Aplica un movimiento. Recibe el ID del jugador y una lista de coordenadas.
        """
        for x, y in moves:
            # Casteamos a int nativo en caso de que numpy devuelva numpy.int64
            x, y = int(x), int(y)
            if self.grid[x, y] == 0:
                self.grid[x, y] = player_id
            else:
                raise ValueError(f"Movimiento inválido: la casilla ({x},{y}) está ocupada.")
        self.turn_count += 1

    def step(self, player_id, move):
        """
        Aplica un movimiento unitario (x, y) como en Gym.
        Retorna: (reward, done, is_valid)
        """
        x, y = move
        if self.grid[x, y] != 0:
            return -10.0, False, False # Recompensa altamente negativa por movimiento inválido
            
        self.grid[x, y] = player_id
        
        if self.check_victory(player_id):
            return 1.0, True, True  # Victoria
            
        if self.is_full():
            return 0.0, True, True  # Empate
            
        return 0.0, False, True     # Continua el juego

    def check_victory(self, player_id):
        """
        Verifica si hay 6 (o más) fichas alineadas horizontal, vertical o diagonalmente
        para el jugador especificado.
        """
        # Comprobar filas
        for i in range(self.size):
            row = self.grid[i, :]
            if self._check_line(row, player_id):
                return True
            
        # Comprobar columnas
        for j in range(self.size):
            col = self.grid[:, j]
            if self._check_line(col, player_id):
                return True
                
        # Comprobar diagonales (principal e inversa)
        # En una matriz de 19x19, las diagonales válidas van de offset -13 a 13
        # porque necesitamos al menos 6 elementos.
        for offset in range(-(self.size - 6), self.size - 5):
            diag1 = np.diagonal(self.grid, offset=offset)
            if self._check_line(diag1, player_id):
                return True
            
            diag2 = np.diagonal(np.fliplr(self.grid), offset=offset)
            if self._check_line(diag2, player_id):
                return True

        return False

    def _check_line(self, line, player_id):
        """Busca si hay al menos 6 fichas consecutivas de `player_id` en un vector 1D."""
        count = 0
        for val in line:
            if val == player_id:
                count += 1
                if count >= 6:
                    return True
            else:
                count = 0
        return False

## 2. Heurísticas propias de Connect6
Victoria y bloqueo inmediato, y puntuación táctica de jugadas.

In [ ]:
"""
Heurísticas propias del juego Connect6.

Estas funciones implementan las "estrategias propias del juego" que pide el
enunciado: detectar situaciones borde (ganar en una jugada, bloquear la victoria
del rival) y puntuar jugadas por su valor táctico. Se usan en dos lugares:

  1. En el agente (MCTSAgent), como pre-chequeo antes de lanzar el MCTS:
     si hay una jugada ganadora se toma directamente; si el rival amenaza con
     ganar, se bloquea.
  2. Como "playout policy" de respaldo del MCTS cuando todavía no hay una red
     neuronal entrenada (permite probar el MCTS sin TensorFlow).
"""

# Direcciones (horizontal, vertical, diagonal \, diagonal /)
_DIRECTIONS = [(1, 0), (0, 1), (1, 1), (1, -1)]


def winning_cells(board, player_id, candidates=None):
    """
    Devuelve las casillas vacías donde `player_id` GANA de inmediato al colocar
    una ficha (completa 6 en línea). Si se le pasa `player_id` del rival, sirve
    para detectar las casillas que hay que bloquear.
    """
    if candidates is None:
        candidates = board.get_candidate_moves(radius=1)
    wins = []
    for (x, y) in candidates:
        if board.grid[x, y] == 0 and board.wins_at(player_id, x, y):
            wins.append((x, y))
    return wins


def _line_score(board, player_id, x, y):
    """
    Puntúa una casilla vacía para `player_id` según cuántas fichas propias y
    del rival hay alineadas a su alrededor. Premia extender líneas propias y
    (con algo menos de peso) cortar líneas del rival.
    """
    n = board.size
    opponent = 2 if player_id == 1 else 1
    score = 0.0

    for dx, dy in _DIRECTIONS:
        own = 0
        opp = 0
        # Ventana de 5 casillas a cada lado a lo largo de la dirección.
        for sign in (1, -1):
            for step in range(1, 6):
                i, j = x + sign * dx * step, y + sign * dy * step
                if not (0 <= i < n and 0 <= j < n):
                    break
                v = board.grid[i, j]
                if v == player_id:
                    # Fichas propias más cercanas valen más.
                    own += (6 - step)
                elif v == opponent:
                    opp += (6 - step)
                    break  # una ficha rival corta nuestra línea en esa dirección
                else:
                    break  # casilla vacía: dejamos de contar consecutivas
        score += own * 1.0 + opp * 0.8  # atacar y defender

    # Sesgo suave hacia el centro del tablero.
    c = n / 2.0
    score += 1.0 - (abs(x - c) + abs(y - c)) / (2 * n)
    return score


def rank_moves(board, player_id, candidates=None, top_k=None):
    """
    Ordena las jugadas candidatas de mejor a peor según la heurística.
    Antepone cualquier jugada ganadora inmediata y luego los bloqueos.
    Devuelve una lista de tuplas (x, y).
    """
    if candidates is None:
        candidates = board.get_candidate_moves(radius=1)

    opponent = 2 if player_id == 1 else 1
    my_wins = set(winning_cells(board, player_id, candidates))
    blocks = set(winning_cells(board, opponent, candidates))

    def priority(move):
        x, y = move
        base = _line_score(board, player_id, x, y)
        if move in my_wins:
            base += 1e6  # ganar ya
        elif move in blocks:
            base += 1e5  # impedir que el rival gane
        return base

    ranked = sorted(candidates, key=priority, reverse=True)
    if top_k is not None:
        ranked = ranked[:top_k]
    return ranked


def best_heuristic_move(board, player_id, rng=None):
    """
    Devuelve la mejor jugada según la heurística. Se usa como playout policy de
    respaldo (sin red neuronal). `rng` opcional para desempatar con algo de azar
    entre las mejores, dando variedad a las simulaciones del MCTS.
    """
    ranked = rank_moves(board, player_id)
    if not ranked:
        return None
    if rng is None:
        return ranked[0]
    # Elige entre las 3 mejores para no ser totalmente determinista.
    k = min(3, len(ranked))
    return ranked[rng.randrange(k)]

## 3. Motor MCTS con selección UCB1
Búsqueda de Monte Carlo con playout policy **conectable** (red neuronal o heurística).

In [ ]:
"""
Monte Carlo Tree Search (MCTS) para Connect6.

Implementa el algoritmo MCTS descrito en el capítulo 6 de Russell & Norvig,
con las cuatro fases clásicas:

    Selección  -> Expansión -> Simulación (playout) -> Retropropagación

La política de SELECCIÓN es UCB1 (la fórmula UCT de la pág. 209 del libro):

        UCB1(hijo) =  Q(hijo)  +  C * sqrt( ln N(padre) / N(hijo) )

donde:
    Q(hijo) = W(hijo) / N(hijo)   valor promedio (explotación)
    C                              constante de exploración (~sqrt(2))
    N(padre), N(hijo)             número de visitas

La política de SIMULACIÓN (playout) es CONECTABLE (`playout_policy`). Por
defecto usa una heurística de Connect6, pero el proyecto la reemplaza por la
recomendación de una red neuronal profunda entrenada con aprendizaje por
refuerzo (ver ai/dqn_agent.py). Ésa es la combinación MCTS + Deep RL que pide
el enunciado (idea de AlphaGo): en la simulación, cada jugador ya no juega al
azar, sino la mejor jugada que le indica el modelo.

Todo este módulo es Python + numpy puro: se puede ejecutar y verificar SIN
TensorFlow usando la playout policy heurística.
"""

import math
import random


BLACK, WHITE = 1, 2


def _other(player):
    return WHITE if player == BLACK else BLACK


class GameState:
    """
    Estado de una partida de Connect6 pensado para la búsqueda.

    Modela la regla de las fichas: el primer jugador (negras) coloca 1 ficha en
    su primer turno y luego cada jugador coloca 2 fichas por turno. Para el árbol
    trabajamos con jugadas de UNA ficha (plies), llevando la cuenta de cuántas
    fichas le quedan al jugador en el turno actual (`stones_left`). Cuando llegan
    a 0, el turno pasa al rival con 2 fichas.
    """

    def __init__(self, board, to_move, stones_left, winner=None):
        self.board = board            # Connect6Board
        self.to_move = to_move        # jugador que debe mover (1 o 2)
        self.stones_left = stones_left
        self.winner = winner          # None, BLACK, WHITE

    @classmethod
    def initial(cls, board=None):
        """Estado inicial: negras al centro, 1 ficha en el primer turno."""
        if board is None:
            board = Connect6Board()
        return cls(board, to_move=BLACK, stones_left=1)

    def legal_moves(self, radius=1):
        return self.board.get_candidate_moves(radius=radius)

    def is_terminal(self):
        return self.winner is not None or self.board.is_full()

    def reward(self):
        """+1 si ganan negras, -1 si ganan blancas, 0 en empate/no terminal."""
        if self.winner == BLACK:
            return 1.0
        if self.winner == WHITE:
            return -1.0
        return 0.0

    def play(self, move):
        """Devuelve un NUEVO estado tras colocar una ficha en `move`."""
        x, y = int(move[0]), int(move[1])
        new_board = self.board.clone()
        won = new_board.wins_at(self.to_move, x, y)
        new_board.grid[x, y] = self.to_move

        winner = self.to_move if won else None
        stones_left = self.stones_left - 1
        to_move = self.to_move
        if winner is None and stones_left == 0:
            to_move = _other(self.to_move)
            stones_left = 2
        return GameState(new_board, to_move, stones_left, winner)


class MCTSNode:
    """Nodo del árbol de búsqueda."""

    __slots__ = ("state", "parent", "move", "player_just_moved",
                 "children", "untried_moves", "N", "W")

    def __init__(self, state, parent=None, move=None, candidates=None):
        self.state = state
        self.parent = parent
        self.move = move  # ficha (x, y) que llevó del padre a este nodo
        # Jugador que hizo la jugada `move` para llegar aquí:
        self.player_just_moved = parent.state.to_move if parent is not None else None
        self.children = {}
        self.untried_moves = list(candidates) if candidates is not None else None
        self.N = 0    # visitas
        self.W = 0.0  # suma de recompensas (desde la óptica de player_just_moved)

    def q_value(self):
        return self.W / self.N if self.N > 0 else 0.0

    def is_fully_expanded(self):
        return self.untried_moves is not None and len(self.untried_moves) == 0

    def ucb1_child(self, c):
        """Selecciona el hijo que maximiza la fórmula UCB1 (UCT)."""
        log_n = math.log(self.N) if self.N > 0 else 0.0
        best, best_score = None, -float("inf")
        for child in self.children.values():
            exploit = child.q_value()                       # Q(hijo)
            explore = c * math.sqrt(log_n / child.N)         # término de exploración
            score = exploit + explore
            if score > best_score:
                best_score, best = score, child
        return best


class MCTS:
    """
    Búsqueda de Monte Carlo. `playout_policy(board, player_id, rng) -> (x, y)`
    decide la jugada durante la simulación. Si es None se usa la heurística.
    """

    def __init__(self, playout_policy=None, n_simulations=200, c=1.4,
                 rollout_depth=50, candidate_radius=1, max_candidates=24,
                 rng=None):
        self.playout_policy = playout_policy
        self.n_simulations = n_simulations
        self.c = c
        self.rollout_depth = rollout_depth
        self.candidate_radius = candidate_radius
        self.max_candidates = max_candidates
        self.rng = rng or random.Random()

    # ---- utilidades ----
    def _candidates(self, state):
        moves = state.legal_moves(radius=self.candidate_radius)
        if self.max_candidates and len(moves) > self.max_candidates:
            # Enfoca la búsqueda en las mejores jugadas según la heurística.
            moves = rank_moves(state.board, state.to_move, moves,
                               top_k=self.max_candidates)
        return moves

    # ---- 4 fases del MCTS ----
    def search(self, root_state):
        """Ejecuta las simulaciones y devuelve la MEJOR ficha (x, y)."""
        root = MCTSNode(root_state, candidates=self._candidates(root_state))
        if not root.untried_moves:
            return None

        for _ in range(self.n_simulations):
            node = self._select(root)
            if not node.state.is_terminal() and node.untried_moves:
                node = self._expand(node)
            reward = self._simulate(node.state)
            self._backpropagate(node, reward)

        # "Robust child": la jugada más visitada (la más fiable).
        best = max(root.children.values(), key=lambda ch: ch.N)
        return best.move

    def _select(self, node):
        """Baja por el árbol usando UCB1 mientras el nodo esté expandido."""
        while (not node.state.is_terminal()
               and node.untried_moves is not None
               and len(node.untried_moves) == 0
               and node.children):
            node = node.ucb1_child(self.c)
        return node

    def _expand(self, node):
        """Añade un hijo para una jugada no probada."""
        move = node.untried_moves.pop(self.rng.randrange(len(node.untried_moves)))
        child_state = node.state.play(move)
        child = MCTSNode(child_state, parent=node, move=move,
                         candidates=self._candidates(child_state)
                         if not child_state.is_terminal() else [])
        node.children[move] = child
        return child

    def _simulate(self, state):
        """
        Playout: simula la partida hasta un estado terminal (o hasta
        `rollout_depth`) usando la playout policy. Devuelve la recompensa final
        desde la óptica de las negras (+1 gana negras, -1 gana blancas).
        """
        board = state.board.clone()
        to_move = state.to_move
        stones_left = state.stones_left
        winner = state.winner

        depth = 0
        while winner is None and depth < self.rollout_depth:
            if board.is_full():
                break
            move = self._playout_move(board, to_move)
            if move is None:
                break
            x, y = int(move[0]), int(move[1])
            if board.wins_at(to_move, x, y):
                winner = to_move
            board.grid[x, y] = to_move
            stones_left -= 1
            if winner is None and stones_left == 0:
                to_move = _other(to_move)
                stones_left = 2
            depth += 1

        if winner == BLACK:
            return 1.0
        if winner == WHITE:
            return -1.0
        return 0.0

    def _playout_move(self, board, player_id):
        # 1) Si hay jugada ganadora inmediata, tomarla (playout "inteligente").
        wins = winning_cells(board, player_id)
        if wins:
            return wins[0]
        # 2) Bloquear victoria inmediata del rival.
        blocks = winning_cells(board, _other(player_id))
        if blocks:
            return blocks[0]
        # 3) Política de simulación: red neuronal si existe, si no heurística.
        if self.playout_policy is not None:
            move = self.playout_policy(board, player_id, self.rng)
            if move is not None:
                return move
        return best_heuristic_move(board, player_id, self.rng)

    def _backpropagate(self, node, reward):
        """
        Sube la recompensa por el camino. La recompensa está en la óptica de las
        negras; para cada nodo la convertimos a la óptica del jugador que movió
        para llegar a él (por eso el signo depende de `player_just_moved`).
        """
        while node is not None:
            node.N += 1
            if node.player_just_moved == BLACK:
                node.W += reward
            elif node.player_just_moved == WHITE:
                node.W -= reward
            node = node.parent

## 4. Agente aleatorio (oponente de entrenamiento y baseline)

In [ ]:
import random

class RandomAgent:
    def __init__(self, player_id):
        self.player_id = player_id

    def get_action(self, board, is_first_turn=False):
        """
        Calcula el siguiente movimiento.
        Retorna 1 coordenada si es el primer turno del juego, o 2 coordenadas para el resto.
        """
        valid_moves = board.get_valid_moves()
        num_stones = 1 if is_first_turn else 2
        
        if len(valid_moves) < num_stones:
            # Retorna lo que quede si no hay suficiente espacio para la cantidad solicitada
            chosen = valid_moves
        else:
            chosen = random.sample(valid_moves, num_stones)

        # Casteo a int nativo (get_valid_moves devuelve numpy.int64).
        return [(int(x), int(y)) for x, y in chosen]

## 5. Red neuronal profunda (DQN)
CNN que estima el valor Q de cada casilla. Se entrena por refuerzo y luego se usa como *playout policy* del MCTS.

In [ ]:
"""
Agente de aprendizaje por refuerzo con red neuronal profunda (Deep Q-Network).

La red es una CNN que, dada la posición del tablero desde la óptica de un
jugador, estima el valor Q de colocar una ficha en cada una de las 361
intersecciones. Entrenada por refuerzo (Q-learning con repetición de
experiencias y red objetivo), aprende a recomendar buenas jugadas.

Su salida se usa de dos formas:
  * Como agente directo (método act / get_action), útil para entrenar.
  * Como PLAYOUT POLICY del MCTS (método policy): durante la simulación del
    árbol, cada jugador coloca la ficha que la red recomienda en vez de una al
    azar. Ésa es la integración MCTS + Deep RL que pide el enunciado.
"""

import numpy as np
import tensorflow as tf
from tensorflow import keras
from collections import deque
import random


class DQNAgent:
    def __init__(self, player_id, state_shape=(19, 19, 1), n_actions=361,
                 learning_rate=0.001, gamma=0.95, epsilon=1.0,
                 epsilon_min=0.01, epsilon_decay=0.995):
        self.player_id = player_id
        self.state_shape = state_shape
        self.n_actions = n_actions
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.learning_rate = learning_rate

        self.replay_buffer = deque(maxlen=10000)

        self.model = self._build_model()
        self.target_model = self._build_model()
        self.update_target_network()

        self.loss_fn = keras.losses.Huber()
        self.optimizer = keras.optimizers.Adam(learning_rate=self.learning_rate)

    def _build_model(self):
        """
        CNN para extraer patrones espaciales del tablero.
        Se construye con keras.Input (API funcional) para ser compatible tanto
        con Keras 2 como con Keras 3 (el que trae Colab): en Keras 3
        `InputLayer(input_shape=...)` ya no es válido.
        """
        inputs = keras.Input(shape=self.state_shape)
        x = keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
        x = keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
        x = keras.layers.Flatten()(x)
        x = keras.layers.Dense(256, activation='relu')(x)
        outputs = keras.layers.Dense(self.n_actions)(x)
        return keras.Model(inputs=inputs, outputs=outputs)

    def update_target_network(self):
        self.target_model.set_weights(self.model.get_weights())

    def remember(self, state, action, reward, next_state, done):
        self.replay_buffer.append((state, action, reward, next_state, done))

    def decay_epsilon(self):
        """Reduce epsilon UNA vez por episodio (no por cada ficha)."""
        if self.epsilon > self.epsilon_min:
            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

    # ------------------------------------------------------------------
    # Selección de jugada
    # ------------------------------------------------------------------
    def _q_values(self, board, player_id):
        state = board.get_state(player_id)
        state_batch = np.expand_dims(state, axis=0)
        # model(x) (llamada directa) es mucho más rápido que model.predict()
        # para lotes pequeños: predict() tiene gran sobrecarga por llamada.
        return self.model(state_batch, training=False).numpy()[0]

    def act(self, board, training=True):
        """Escoge una única acción (x, y) con Epsilon-Greedy y máscara de válidos."""
        valid_moves = board.get_valid_moves()
        if not valid_moves:
            return None

        valid_actions_flat = [x * board.size + y for x, y in valid_moves]

        if training and np.random.rand() <= self.epsilon:
            action_flat = random.choice(valid_actions_flat)
        else:
            q_values = self._q_values(board, self.player_id)
            masked_q_values = np.full(self.n_actions, -np.inf)
            masked_q_values[valid_actions_flat] = q_values[valid_actions_flat]
            action_flat = int(np.argmax(masked_q_values))

        x, y = divmod(action_flat, board.size)
        return (int(x), int(y))

    def recommend_move(self, board, player_id, candidates=None):
        """
        Devuelve la MEJOR jugada según la red para `player_id`, restringida a las
        casillas candidatas (poda de vecindad). Es la base de la playout policy.
        """
        if candidates is None:
            candidates = board.get_candidate_moves(radius=1)
        if not candidates:
            return None
        q_values = self._q_values(board, player_id)
        best, best_q = None, -np.inf
        for (x, y) in candidates:
            q = q_values[x * board.size + y]
            if q > best_q:
                best_q, best = q, (int(x), int(y))
        return best

    def policy(self, board, player_id, rng=None, epsilon=0.1):
        """
        Playout policy para el MCTS: casi siempre la jugada recomendada por la
        red; con probabilidad `epsilon` una candidata al azar (da variedad a las
        simulaciones). Firma compatible con MCTS: (board, player_id, rng).
        """
        candidates = board.get_candidate_moves(radius=1)
        if not candidates:
            return None
        if rng is not None and rng.random() < epsilon:
            return candidates[rng.randrange(len(candidates))]
        return self.recommend_move(board, player_id, candidates)

    def get_action(self, board, is_first_turn=False, training=False):
        """Interfaz de turno: retorna las N fichas (1 en el primer turno, si no 2)."""
        num_stones = 1 if is_first_turn else 2
        moves = []
        for _ in range(num_stones):
            move = self.act(board, training)
            if move:
                moves.append(move)
                board.grid[move[0], move[1]] = self.player_id  # ocupar temporal
        for m in moves:  # limpiar el rastro (main.py llamará a apply_move)
            board.grid[m[0], m[1]] = 0
        return moves

    # ------------------------------------------------------------------
    # Entrenamiento
    # ------------------------------------------------------------------
    def train_step(self, batch_size=32):
        if len(self.replay_buffer) < batch_size:
            return 0.0

        indices = np.random.choice(len(self.replay_buffer), batch_size, replace=False)
        batch = [self.replay_buffer[i] for i in indices]

        states, actions, rewards, next_states, dones = map(np.array, zip(*batch))
        rewards = rewards.astype(np.float32)
        dones = dones.astype(np.float32)

        next_Q_values = self.target_model(next_states, training=False).numpy()
        max_next_Q_values = np.max(next_Q_values, axis=1)
        target_Q_values = rewards + (1.0 - dones) * self.gamma * max_next_Q_values

        mask = tf.one_hot(actions, self.n_actions)

        with tf.GradientTape() as tape:
            all_Q_values = self.model(states)
            Q_values = tf.reduce_sum(all_Q_values * mask, axis=1, keepdims=True)
            target = tf.expand_dims(target_Q_values, axis=1)
            loss = tf.reduce_mean(self.loss_fn(target, Q_values))

        grads = tape.gradient(loss, self.model.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.model.trainable_variables))

        return float(loss.numpy())

    # ------------------------------------------------------------------
    # Persistencia (formato .keras, el nativo de Keras 3)
    # ------------------------------------------------------------------
    def save(self, filename):
        self.model.save(filename)

    def load(self, filename):
        self.model = keras.models.load_model(filename)
        self.update_target_network()

## 6. Agente principal: MCTS + red neuronal + heurísticas

In [ ]:
"""
Agente de juego de Connect6 basado en MCTS + red neuronal + heurísticas.

Es el agente principal que pide el enunciado. Para cada turno decide 1 ficha
(primer turno de negras) o 2 fichas con la siguiente lógica:

    1. Apertura: si el tablero está vacío, juega al centro (9, 9).
    2. Victoria inmediata: si puede completar 6 en línea, lo hace.
    3. Bloqueo: si el rival amenaza con ganar en su próxima ficha, lo bloquea.
    4. En otro caso, ejecuta MCTS (selección UCB1) para elegir la ficha; durante
       la simulación del MCTS se usa la red neuronal como playout policy (o la
       heurística si no se ha entrenado ninguna red).

El método `get_action` devuelve la lista de fichas del turno, en el mismo
formato que espera main.py y el cliente de Connect6 Arena.
"""

import random


BLACK, WHITE = 1, 2


class MCTSAgent:
    def __init__(self, player_id, dqn_agent=None, n_simulations=200, c=1.4,
                 rollout_depth=50, max_candidates=24, use_heuristics=True,
                 rng=None):
        self.player_id = player_id
        self.opponent = WHITE if player_id == BLACK else BLACK
        self.dqn_agent = dqn_agent
        self.use_heuristics = use_heuristics
        self.rng = rng or random.Random()

        playout_policy = dqn_agent.policy if dqn_agent is not None else None
        self.mcts = MCTS(
            playout_policy=playout_policy,
            n_simulations=n_simulations,
            c=c,
            rollout_depth=rollout_depth,
            max_candidates=max_candidates,
            rng=self.rng,
        )

    def _choose_stone(self, board, stones_left):
        """Elige UNA ficha aplicando heurísticas y, si hace falta, MCTS."""
        if self.use_heuristics:
            # 2. Victoria inmediata propia.
            my_wins = winning_cells(board, self.player_id)
            if my_wins:
                return my_wins[0]
            # 3. Bloqueo de victoria inmediata del rival.
            opp_wins = winning_cells(board, self.opponent)
            if opp_wins:
                return opp_wins[0]

        # 4. Búsqueda MCTS.
        state = GameState(board.clone(), to_move=self.player_id,
                          stones_left=stones_left)
        move = self.mcts.search(state)
        if move is None:  # tablero sin candidatos (borde): cae a cualquier válida
            valid = board.get_valid_moves()
            move = self.rng.choice(valid) if valid else None
        return move

    def get_action(self, board, is_first_turn=False):
        """Devuelve la lista de fichas (1 o 2) a jugar este turno."""
        num_stones = 1 if is_first_turn else 2

        # 1. Apertura al centro.
        if is_first_turn and int((board.grid != 0).sum()) == 0:
            c = board.size // 2
            return [(c, c)]

        moves = []
        work = board.clone()  # tablero de trabajo para decidir las 2 fichas
        for k in range(num_stones):
            if work.is_full():
                break
            stone = self._choose_stone(work, stones_left=num_stones - k)
            if stone is None:
                break
            stone = (int(stone[0]), int(stone[1]))
            moves.append(stone)
            work.grid[stone[0], stone[1]] = self.player_id
            # Si esta ficha ya gana, no hace falta la segunda.
            if work.wins_at(self.player_id, stone[0], stone[1]):
                break
        return moves

## 7. Entrenamiento por refuerzo de la red

La red aprende jugando contra un agente aleatorio (Q-learning con repetición de
experiencias y red objetivo). Ajusta `EPISODES`: 200–500 dan una demostración
razonable; usa varios miles para un agente fuerte (más lento).


In [ ]:
import numpy as np
import time

def train_dqn(episodes=300, batch_size=32, target_update=5, verbose_every=10):
    agent = DQNAgent(player_id=1, epsilon=1.0, epsilon_min=0.1, epsilon_decay=0.995)
    opponent = RandomAgent(player_id=2)
    rewards_hist = []
    t0 = time.time()

    for episode in range(episodes):
        board = Connect6Board()
        done = False
        turn = 0
        total_reward = 0.0
        state = None
        action_flat = None

        while not done:
            num_stones = 1 if turn == 0 else 2
            for _ in range(num_stones):
                state = board.get_state(player_id=1)
                move = agent.act(board, training=True)
                if move is None:
                    done = True
                    break
                action_flat = move[0] * board.size + move[1]
                reward, done, _ = board.step(player_id=1, move=move)
                next_state = board.get_state(player_id=1)
                agent.remember(state, action_flat, reward, next_state, done)
                total_reward += reward
                agent.train_step(batch_size)
                if done:
                    break
            if done:
                break

            white = opponent.get_action(board, is_first_turn=False)
            if not white:
                break
            for w in white:
                _, done, _ = board.step(player_id=2, move=w)
                if done:
                    next_state = board.get_state(player_id=1)
                    agent.remember(state, action_flat, -1.0, next_state, done)
                    total_reward -= 1.0
                    break
            turn += 1

        agent.decay_epsilon()
        if episode % target_update == 0:
            agent.update_target_network()
        rewards_hist.append(total_reward)

        if (episode + 1) % verbose_every == 0:
            avg = np.mean(rewards_hist[-verbose_every:])
            print(f"Episodio {episode+1}/{episodes} | recompensa media (últimos {verbose_every}): "
                  f"{avg:+.2f} | epsilon: {agent.epsilon:.3f} | {time.time()-t0:.0f}s")

    print(f"\nEntrenamiento terminado en {time.time()-t0:.0f}s")
    return agent, rewards_hist

EPISODES = 300  # súbelo (p.ej. 3000) para un agente más fuerte
dqn_agent, rewards_hist = train_dqn(episodes=EPISODES)

### Curva de recompensa

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def moving_avg(x, w=20):
    x = np.array(x, dtype=float)
    if len(x) < w:
        return x
    return np.convolve(x, np.ones(w)/w, mode="valid")

plt.figure(figsize=(9, 4))
plt.plot(rewards_hist, alpha=0.3, label="recompensa por episodio")
plt.plot(range(len(moving_avg(rewards_hist)) ), moving_avg(rewards_hist), label="media móvil (20)")
plt.xlabel("episodio"); plt.ylabel("recompensa"); plt.legend(); plt.title("Entrenamiento del DQN")
plt.grid(alpha=0.3); plt.show()

## 8. Guardar el modelo entrenado

In [ ]:
MODEL_PATH = "dqn_model.keras"
dqn_agent.save(MODEL_PATH)
print("Modelo guardado en", MODEL_PATH)

## 9. Evaluación: ¿ayuda la red al MCTS?

Comparamos tres agentes jugando de negras contra un rival aleatorio:

* **MCTS + red neuronal** (el agente completo),
* **MCTS con playout heurístico** (sin red),
* la **red sola** (sin MCTS).


In [ ]:
def play_game(black_agent, white_agent, max_turns=200):
    board = Connect6Board()
    for turn in range(max_turns):
        bm = black_agent.get_action(board, is_first_turn=(turn == 0))
        if not bm:
            return 0
        board.apply_move(1, bm)
        if board.check_victory(1):
            return 1
        wm = white_agent.get_action(board, is_first_turn=False)
        if not wm:
            return 0
        board.apply_move(2, wm)
        if board.check_victory(2):
            return -1
        if board.is_full():
            return 0
    return 0

def win_rate(make_black, n_games=10):
    wins = 0
    for _ in range(n_games):
        r = play_game(make_black(), RandomAgent(player_id=2))
        wins += 1 if r == 1 else 0
    return wins / n_games

# Nota: con red neuronal, cada simulación llama a la red varias veces, así que
# subir n_simulations o N hace la evaluación bastante más lenta.
N = 6
mcts_nn   = lambda: MCTSAgent(player_id=1, dqn_agent=dqn_agent, n_simulations=40, rollout_depth=30)
mcts_only = lambda: MCTSAgent(player_id=1, dqn_agent=None,      n_simulations=40, rollout_depth=30)

# La red sola: pequeño wrapper con get_action (epsilon=0)
class _NNOnly:
    def __init__(self):
        self.a = dqn_agent
        self.a.epsilon = 0.0
    def get_action(self, board, is_first_turn=False):
        return self.a.get_action(board, is_first_turn=is_first_turn, training=False)

print("MCTS + red neuronal vs aleatorio:", win_rate(mcts_nn, N))
print("MCTS heurístico   vs aleatorio:", win_rate(mcts_only, N))
print("Red sola          vs aleatorio:", win_rate(lambda: _NNOnly(), N))

## 10. Partida de demostración (agente de negras vs aleatorio)

In [ ]:
def print_board(board):
    print("   " + " ".join(f"{i:2}" for i in range(board.size)))
    for i in range(board.size):
        row = f"{i:2} "
        for j in range(board.size):
            v = board.grid[i, j]
            row += " · " if v == 0 else (" X " if v == 1 else " O ")
        print(row)
    print()

board = Connect6Board()
black = MCTSAgent(player_id=1, dqn_agent=dqn_agent, n_simulations=60, rollout_depth=30)
white = RandomAgent(player_id=2)
winner = None
for turn in range(200):
    bm = black.get_action(board, is_first_turn=(turn == 0)); board.apply_move(1, bm)
    if board.check_victory(1): winner = "Negras (agente)"; break
    wm = white.get_action(board, is_first_turn=False); board.apply_move(2, wm)
    if board.check_victory(2): winner = "Blancas (aleatorio)"; break
    if board.is_full(): break
print_board(board)
print("Resultado:", winner or "empate / tope de turnos")

## 11. Descargar el modelo a tu PC
Bájalo y colócalo en la carpeta `proyecto2` local; `main.py` lo usará automáticamente.

In [ ]:
try:
    from google.colab import files
    files.download("dqn_model.keras")
except Exception as e:
    print("Descarga manual desde el panel de Archivos. Detalle:", e)

## 12. Siguiente paso: conectar a Connect6 Arena

Para jugar contra otros agentes o un humano se usa la plataforma **Connect6 Arena**
(gRPC). El agente ya expone `get_action(board, is_first_turn)` que devuelve las
fichas del turno; sólo hay que traducir los mensajes gRPC del *starter-kit* de
Laura Parilli a llamadas a ese método (ver `network/client.py` y `Leeme.txt`).
